# Econometrics HW 1


##### Initial analysis: 

In [24]:
import numpy as np
import pandas as pd
from spicy import stats
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
df = pd.read_csv("my_data.csv")
print(df.shape)

(100, 5)


In [4]:
print(df.head(1))
print(f"\n")
print(df.iloc[0])

      Y    X1     X2  D  Z
0 -2.11  4.53  19.47  0  A


Y     -2.11
X1     4.53
X2    19.47
D         0
Z         A
Name: 0, dtype: object


##### Задача 1

Найди минимальное значение Y в твоей выборке.


In [5]:
min_y = df['Y'].min()
print(min_y)

-4.9


##### Задача 2

Найди дисперсию X2 в твоей выборке.

In [6]:
var_x2 = df['X2'].var()
print(var_x2)

18.41502297979798


##### Задача 3

Найди среднее значение Y среди наблюдений, где X1 больше 5.

In [7]:
mean_y = df.loc[df['X1'] > 5, 'Y'].mean()
print(mean_y)

0.8838181818181818


##### Задача 4
Проверь при помощи t-теста гипотезу о том, что матожидание Y равно -0.1. В ответ введи p-value. 

In [9]:
t_stat, p_value = stats.ttest_1samp(df["Y"].dropna(), popmean=-0.1)

print(f"p-value: {p_value}")

p-value: 0.8151418174320324


##### Задача 5
Найди корреляцию Y и X1.

In [10]:
corr_yx =  df['Y'].corr(df['X1'])
print(corr_yx)

0.7094026065688804


##### Задача 6
Найди коэффициент при X2 в регрессии Y на X1 и X2.

In [ ]:
reg = LinearRegression().fit(df[["X1", "X2"]], df["Y"])
b2 = reg.coef_[1]  # индекс 1 соответствует X2
print(b2)

0.07174417943779907


##### Задача 7
Найди R^2 в регрессии Y на X1 и X2.

In [13]:
X = df[['X1', 'X2']]
y = df['Y']

model = LinearRegression()
model.fit(X, y)

r_squared = model.score(X, y)
print(f'R^2: {r_squared:.4f}')

R^2: 0.5390


##### Задача 8
Найди t-статистику для гипотезы о равенстве коэффициента при X1 нулю в регрессии Y на X1 и X2.

In [17]:
X = df[['X1', 'X2']]
y = df['Y']

# 2. Обязательно добавляем константу (свободный член / intercept)
X = sm.add_constant(X)

# 3. Обучаем модель OLS
model = sm.OLS(y, X).fit()

# 4. Извлекаем t-статистику конкретно для X1
t_stat_x1 = model.tvalues['X1']

print(f't-статистика для X1: {t_stat_x1:.4f}')

# Вывод полной таблицы результатов (включая t-статистику и p-value)
print(model.summary())

t-статистика для X1: 10.1391
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.539
Model:                            OLS   Adj. R-squared:                  0.530
Method:                 Least Squares   F-statistic:                     56.72
Date:                Sun, 15 Nov 2026   Prob (F-statistic):           4.87e-17
Time:                        21:31:02   Log-Likelihood:                -151.24
No. Observations:                 100   AIC:                             308.5
Df Residuals:                      97   BIC:                             316.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -6.7346  

##### Задача 9
Найди оценку дисперсии ошибок в регрессии Y на X1 и X2.

In [18]:
X = df[['X1', 'X2']]
y = df['Y']

# Добавляем константу (свободный член)
X = sm.add_constant(X)

# Строим модель и получаем оценку дисперсии ошибок
model = sm.OLS(y, X).fit()
error_variance = model.mse_resid

print(f"Оценка дисперсии ошибок: {error_variance}")

Оценка дисперсии ошибок: 1.2427331045659384


##### Задача 10
Найди коэффициент при X1 в регрессии Y на X1 и X2 без константы(!).

In [ ]:
X = df[["X1", "X2"]].to_numpy()
y = df["Y"].to_numpy()

beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
coef_x1 = beta[0]

print(coef_x1)

0.1517271927670625


##### Задача 11
Найди коэффициент при D в регрессии Y на X2 и D.

In [21]:
# df — ваш DataFrame со столбцами 'Y', 'X2', 'D'
X = sm.add_constant(df[["X2", "D"]])
model = sm.OLS(df["Y"], X).fit()

coef_d = model.params["D"]
print(coef_d)

0.1251592219876822


##### Задача 12
Включи в регрессию Y на X1 и X2 дамми-переменные для категорий Z, взяв B в качестве базовой категории. Найди коэффициент при дамми для категории C.

In [22]:
dummies = pd.get_dummies(df['Z'], prefix='Z', dtype=float).drop(columns=['Z_B'])

# Формируем матрицу признаков X и целевую переменную y
X = pd.concat([df[['X1', 'X2']], dummies], axis=1)
y = df['Y']

# Обучаем модель
reg = LinearRegression().fit(X, y)

# Находим индекс столбца 'Z_C' и выводим его коэффициент
idx_c = list(X.columns).index('Z_C')
print(f"Коэффициент при дамми для категории C: {reg.coef_[idx_c]}")

Коэффициент при дамми для категории C: 0.3164237602294204


##### Задача 13
Оцени регрессию Y на X1, X2, D и произведение X1*D. Найди оценку коэффициента при X1 для наблюдений с D = 1.

In [25]:
# 1. Оценка модели (запись X1 * D автоматически включает X1, D и их произведение X1:D)
model = smf.ols("Y ~ X1 * D + X2", data=df).fit()

# Вывод полной таблицы результатов
print(model.summary())

# 2. Расчет коэффициента при X1 для наблюдений с D = 1
coef_x1_d1 = model.params["X1"] + model.params["X1:D"]
print(f"Оценка коэффициента при X1 при D = 1: {coef_x1_d1:.4f}")

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.575
Model:                            OLS   Adj. R-squared:                  0.557
Method:                 Least Squares   F-statistic:                     32.16
Date:                Sun, 15 Nov 2026   Prob (F-statistic):           6.16e-17
Time:                        21:35:27   Log-Likelihood:                -147.15
No. Observations:                 100   AIC:                             304.3
Df Residuals:                      95   BIC:                             317.3
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -5.5144      0.871     -6.332      0.0

##### Задача 14
В регрессии Y на X1, X2 и D проверь гипотезу о том, что коэффициенты при X1 и X2 одновременно равны нулю. В ответ введи F-статистику.

In [26]:
# Предположим, ваши данные находятся в DataFrame с именем df
# Пример создания данных:
# df = pd.DataFrame({'Y': [...], 'X1': [...], 'X2': [...], 'D': [...]})

# Строим модель МНК (OLS)
model = smf.ols('Y ~ X1 + X2 + D', data=df).fit()

# Проверяем совместную гипотезу: X1 = 0 и X2 = 0
f_test_result = model.f_test('X1 = 0, X2 = 0')

# Извлекаем значение F-статистики
f_stat = f_test_result.fvalue
print(f"F-статистика: {f_stat:.4f}")

F-статистика: 57.7743


##### Задача 15
Оцени регрессию Y на X1, D и произведение X1*D. Проверь гипотезу о том, что коэффициенты при D и X1*D одновременно равны нулю. В ответ введи F-статистику.
 

In [27]:
df['const'] = 1
df['X1_D'] = df['X1'] * df['D']

X_full = df[['const', 'X1', 'D', 'X1_D']].values
y = df['Y'].values

# Оценка полной модели методом МНК
beta_full = np.linalg.lstsq(X_full, y, rcond=None)[0]
rss_full = np.sum((y - X_full @ beta_full) ** 2)

# Оценка ограниченной модели (только const и X1, так как D=0 и X1*D=0)
X_rest = df[['const', 'X1']].values
beta_rest = np.linalg.lstsq(X_rest, y, rcond=None)[0]
rss_rest = np.sum((y - X_rest @ beta_rest) ** 2)

# Расчет F-статистики
q = 2  # число ограничений (D = 0 и X1*D = 0)
n = len(df)  # число наблюдений
k = X_full.shape[1]  # число параметров в полной модели (4)

f_stat = ((rss_rest - rss_full) / q) / (rss_full / (n - k))
print(f"F-статистика: {f_stat}")

F-статистика: 3.0610281376557924
